# 07. 公開OOFの導入: 点検してから、他の参加者の予測を2段目に加える

**このノートブックの目的:** 06のスタック(自分の5モデル)に、他の参加者が公開したOOFを加え、混ぜるモデルの数と種類を増やす。混ぜる前に、正しく作られているかを点検する方法を決める。

**使うもの:** 公開OOF(他の参加者の成果物なので、このリポジトリには含めない。取得方法はREADMEの実行方法)と、自分の4モデルのOOF・testの予測。

**結論:** 点検に合格した公開OOFを加え、メンバー選び・混ぜ方を調整した最終の226本のスタックで **OOF AUC 0.962266、Private LB 0.96239(52位 / 3,748チーム、上位1.4%)**。06の5モデルのスタック(Private LB 0.96208、上位29%)から+0.00031伸びた。

## 準備

- ライブラリと、共通処理(`src/`の自分で書いたファイル)を読み込む。`src/features.py` はデータの読み込みと特徴量の作成、`src/modeling.py` はモデルの学習と評価。
- train・testを読み込み、目的変数 `satisfaction` を0/1の配列 `y` にする。
- `src/public_oof.py`(公開OOFの点検・形の統一・系統分け)を読み込む。`RUN_COLLECTION = False` のときは、形をそろえて保存した公開OOFをそのまま使う(Trueにすると、取得したままの公開OOFを読み直して形をそろえる)。

In [1]:
import sys

sys.path.insert(0, "../src")  # 共通処理(features.py・modeling.py・public_oof.py)を読み込む
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import features
import modeling

plt.rcParams["font.family"] = "Meiryo"  # グラフの日本語ラベル用
DATA = features.DATA_DIR
train, test = features.load_train_and_test()
y = train[features.TARGET].astype(int).values
print(f"train {len(train):,}行・test {len(test):,}行、満足率 {y.mean():.3f}")
import public_oof

RUN_COLLECTION = False  # Trueにすると、取得したままの公開OOF(data/pub_oof/raw/)を読み直して形をそろえる

train 699,635行・test 299,844行、満足率 0.444


## 1. 公開OOFを点検する

**公開OOFとは:** 他の参加者が公開した、OOFとtestの予測のセット。予測の値だけなので、中のモデルや分け方は公開した人しだい。それでも、OOFが「その行を学習していないモデルの予測」になっていれば、2段目からは混ぜる予測が1本増えるだけで、06の仕組みにそのまま入れられる。

**目的:** 作り方が見えないので、正しく作られていないもの(OOFが甘いもの等)を混ぜると、手元のOOFだけが良く見えてPrivate LBで崩れるおそれがある。混ぜる前に点検する。

**検証方法:** 最初に見つけた124本(cdeotte氏のデータセット)について、次の3つを確かめる。
- 行数・欠損・範囲外の値: trainとtestの各行に、0〜1の予測が1つずつそろっているか。
- 予測値の平均が約0.44か: 満足の割合は約0.44なので、確率として作られていれば平均もそれに近い。ちょうど0.5なら、順位に直して混ぜたもので、中身がほかの公開OOFと重なるおそれがある。
- 単体のAUCが、自分の5モデルのスタック(0.9617)より高すぎないか: それ自体が答えを使って混ぜた結果の可能性がある。

In [2]:
inspection = public_oof.inspect_public_oof(DATA / "pub_oof" / "cdeotte124", y, len(test), np.load(DATA / "oof_realmlp.npy"))
display(inspection["judgement"].value_counts().to_frame("本数"))
inspection.sort_values("auc", ascending=False).head(5)[["name", "auc", "oof_mean", "test_mean", "correlation_with_own_model", "judgement"]].round(4)

,本数
judgement,
OK,123
要確認(高すぎ),1


,name,auc,oof_mean,test_mean,correlation_with_own_model,judgement
93,nj_01,0.9617,0.5000,0.5000,0.9874,要確認(高すぎ)
112,sa_rmlp_v4aux_k10s2,0.9614,0.4434,0.4435,0.9802,OK
111,sa_rmlp_v4aux_k10,0.9614,0.4435,0.4436,0.9801,OK
105,sa_rmlp_B_k10,0.9614,0.4435,0.4436,0.9814,OK
54,gp_realmlp_vb_aux,0.9613,0.4436,0.4437,0.9833,OK


#### 結果と考察

124本のうち1本を除外した(予測値の平均がちょうど0.5で、単体のAUCが自分のスタックより高い)。順位に直してほかのモデルを混ぜたものとみた。残りは形も平均もそろっていた。

## 2. ほかの公開OOFを集め、形をそろえる

**目的:** 1の124本で伸びたので(下の4の表の1)、ほかの参加者が公開したOOFのデータセット(約20)も集める。

**検証方法:** データセットごとにファイルの形(npy・npz・parquet・csv)が違うので、同じ形(`oof_<名前>.npy`・`test_<名前>.npy`、行の順はtrain・testと同じ)に書き直す。次のものは外す。
- 行数・範囲が合わないもの。
- 名前に stack・blend を含むもの(それ自体が2段目の結果。2段目に2段目を重ねると、混ぜ方を学んだ行と採点する行が重なり、OOFが甘くなる)。
- 既存のものとlogitの相関が0.9999以上のもの(同じ予測の写し。同じファイルが別のデータセットに再掲されていることが多い)。

In [3]:
if RUN_COLLECTION:
    candidates = public_oof.read_raw_public_oof(DATA / "pub_oof" / "raw", len(train))
    gathered = public_oof.save_unique_public_oof(candidates, [DATA / "pub_oof" / "cdeotte124"], DATA / "pub_oof" / "extra", len(train), len(test))
    gathered.to_csv(DATA / "pub_oof" / "extra" / "_gather.csv", index=False)
    # 後から公開された11本: ノートブックの出力(data/pub_oof/raw_late/)から、一覧に書いた列を取り出す
    public_oof.extract_late_public_oof(pd.read_csv("../src/late_public_oof_extraction.csv", dtype=str, keep_default_na=False), DATA)
gather_file = DATA / "pub_oof" / "extra" / "_gather.csv"
if gather_file.exists():
    gathered = pd.read_csv(gather_file)
    display(gathered["判定" if "判定" in gathered else "judgement"].value_counts().to_frame("本数"))

,本数
判定,
採用,136
重複,50
除外(2段目の結果),4
形が合わない,1


## 3. 最終のメンバーと系統

最終の226本 = 公開OOF 222本(1の123本+2で足した136本−LightGBM系統48本+後から公開された11本)+自分の4本(RealMLP・TabPFN・XGBoost・CatBoost)。一覧は `src/final_stack_members.csv`(名前・系統・置き場所・出典)。系統は名前から決めた(`public_oof.model_family`)。

In [4]:
members = pd.read_csv("../src/final_stack_members.csv")
members["系統(名前から)"] = members["member_name"].map(public_oof.model_family)
own = members["data_subfolder"] == "own"
members.loc[own, "系統(名前から)"] = members.loc[own, "model_family"]  # 自分のモデルは一覧の系統を使う
print(f"メンバー {len(members)}本(自分 {own.sum()}本・公開OOF {(~own).sum()}本)")
members["model_family"].value_counts().rename({"mlp": "ニューラルネット", "cat": "CatBoost", "xgb": "XGBoost", "tabpfn": "TabPFN",
                                               "tfm": "表の基盤モデル", "other": "その他"}).to_frame("本数")

メンバー 226本(自分 4本・公開OOF 222本)


,本数
model_family,
ニューラルネット,75
XGBoost,51
CatBoost,50
TabPFN,17
その他,12
(後から追加),11
表の基盤モデル,10


## 4. 試したことと結果

公開OOFを加えたあと、メンバー選びと混ぜ方を1つずつ試した。採否は、2段目の評価の分け方を変えた3通りでも確かめた(06と同じ「4つで学び残り1つで採点」のOOF AUC)。

| # | 試したこと | 試した理由 | 結果(OOF AUC) | 判定 |
|---|---|---|---|---|
| 1 | 公開OOF 123本を加えた | 上位との差は、混ぜている他の参加者のモデルの数と種類から来ていると、上位の公開ノートブックで報告されていた | +0.00038(Public LB +0.00043) | 採用 |
| 2 | ほかの公開データセットから136本を足した(計264本) | 1で効いたので、ほかのデータセット(約20)も集めた。191本から写し50本・2段目の結果4本・形が合わない1本を外した | +0.000163(5foldすべてで改善) | 採用 |
| 3 | 系統ごとにまとめて外し、どの系統が効くかを見た | 264本に増え、どの系統が伸びに効いているかが分からなくなった | 外したときの減少: TabPFN −0.000070、ニューラルネット −0.000046、CatBoost −0.000044、表の基盤モデル −0.000034 | 手がかり |
| 4 | 2段目の正則化の強さを7通り比べた | 似た予測が多く、正則化が弱いと重みが似たメンバーの間でぶれやすい | C=0.003が最良(0.0003で−0.000072、0.03で−0.000031) | 採用 |
| 5 | LightGBM系統の49本(自分のLightGBMを含む)をまとめて外した | 3で、LightGBM系統だけは外すとOOFが上がった(+0.000016) | +0.000014(3seedそろって改善。同じ本数をランダムに外すと悪化) | 採用 |
| 6 | 2段目に浅いLightGBMを2割混ぜた | ロジスティック回帰は重み付きで足すだけなので、メンバーどうしの組み合わせで変わる効き方を浅い木で補う | +0.000006(3seedとも改善) | 採用 |
| 7 | 似たメンバーを間引いた | 写しのような似たメンバーが多い | 相関0.999以上(65本)で±0、0.998以上で−0.000127 | 不採用 |
| 8 | 係数がマイナスの105本を外した | 予測を逆向きに使っているように見えた | −0.00015(似たメンバーとの差し引き役だった) | 不採用 |
| 9 | 後から公開されたOOFを1本ずつ足した | コンペ中も新しい公開OOFが出続けた | 1本あたり+0.000004〜+0.000007(既存との相関0.995〜0.999) | 改善したものだけ採用(11本) |
| 10 | 自分のTabM(ニューラルネット)を加えた | 公開OOFに頼らず、自分のモデルで伸ばせるか | ±0.000003以内(最も違う予測だが、単体が約0.0024低く重みが付かない) | 不採用 |

## 5. 最終のスタックを作る

**検証方法:** 226本のOOFをlogitに直し、2段目はロジスティック回帰(C=0.003)8割+浅いLightGBM2割(それぞれ平均0・標準偏差1にそろえて足す)。06と同じ「4つで学び残り1つで採点」でOOF AUCを測り、提出用はtrainの全行で学び直してtestに当てる(浅いLightGBMは5つのseedの平均)。

In [5]:
def load_member_logits(prediction_type):
    """全メンバーの予測(prediction_type: "oof" か "test")を読み、logitにして横に並べる。"""
    columns = []
    for _, member in members.iterrows():
        folder = DATA if member["data_subfolder"] == "own" else DATA / member["data_subfolder"]
        columns.append(modeling.to_logit(np.load(folder / f"{prediction_type}_{member['member_name']}.npy").ravel()).astype(np.float32))
    return np.column_stack(columns)


oof_logits = load_member_logits("oof")
stack = modeling.cross_validate_final_stack(oof_logits, y)
print(f"ロジスティック回帰だけ {roc_auc_score(y, stack['logistic']):.6f} / 浅いLightGBMだけ {roc_auc_score(y, stack['lightgbm']):.6f}")
print(f"最終のスタック(8:2) OOF AUC {roc_auc_score(y, stack['blended']):.7f}")

ロジスティック回帰だけ 0.962260 / 浅いLightGBMだけ 0.962034


最終のスタック(8:2) OOF AUC 0.9622665


In [6]:
test_probability = modeling.fit_final_stack_and_predict(oof_logits, y, load_member_logits("test"), stack["best_iterations"])
submission = pd.DataFrame({"id": test["id"], "satisfaction": test_probability})
submission.to_csv(DATA / "submission_final226.csv", index=False)
print(f"data/submission_final226.csv({len(submission):,}行、予測の平均 {test_probability.mean():.3f})")

data/submission_final226.csv(299,844行、予測の平均 0.496)


## 6. 最終提出の選び方

2本を選んだ。
- **1本目:** CVで最良の226本のスタック(Public LB 0.96182、Private LB 0.96239)。
- **2本目(保険):** 1本目と、他の参加者がPublic LBに合わせて作った公開の提出ファイル(Public LB 0.96213)を、順位で半々に混ぜたもの(Public LB 0.96202、Private LB 0.96236)。公開の提出ファイルはOOFが無くCVで測れないので、混ぜるのは2本目だけにし、比率もPublic LBで選ばず半々に固定した。

**結果:** CVで選んだ1本目がPrivate LBで最良だった。Public LB(約6万行)は0.0003未満の差を見分けられない。Public LBでのチームの順位は513位だったが、Private LBでは52位になった。

## Summary

| 段階 | OOF AUC | Private LB |
|---|---|---|
| 06の5モデルのスタック | 0.961685 | 0.96208(上位29%) |
| 公開OOFを加えた最終の226本 | 0.962266 | **0.96239(52位 / 3,748チーム、上位1.4%)** |

(数値は実験記録の値。このノートブックを再実行すると、自分のXGBoostを学び直した分などで、OOF AUCの小数第6位が少しずれることがある)